### Embeddings + Vector DB (FAISS)

- This is the core of any RAG (Retrieval-Augmented Generation) system.

___

##### Step 1: What Are Embeddings?

- Embeddings convert text → numbers (vectors).

```bash
"Random Forest" → [0.12, -0.9, 0.44, ...]
```
- These vectors allow computers to measure semantic similarity.

### Step 2: Install FAISS (CPU Version)

- FAISS = Facebook AI Similarity Search
- Used for fast vector search.
```bash
pip install faiss-cpu langchain langchain-community
```


___

#### Step 3: Choose Your Embedding Model


- Best free CPU embedding model:
  - sentence-transformers/all-MiniLM-L6-v2

```bash

pip install sentence-transformers
```
___
```python
from langchain_community.embeddings import SentenceTransformerEmbeddings

embed_model = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
```

### Step 4: Embed Your Chunks

- Assume you have chunks from previous step.
```python
embeddings = embed_model.embed_documents([chunk.page_content for chunk in chunks])
print(len(embeddings), len(embeddings[0]))

```

### Step 5: Store Embeddings in FAISS

```python
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(chunks, embed_model)
```

### Step 6: Save & Load Vector Database

```python
vector_db.save_local("faiss_index")

```

- Load
```python
db = FAISS.load_local("faiss_index", embed_model, allow_dangerous_deserialization=True)
```

### Step 7: Query the Vector Database
```python
query = "What is the purpose of cross-validation?"
results = vector_db.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("---")

```

### Step 8: Add the LLM to Create a RAG Pipeline
```python
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="llama3.1")
```

- Now create RAG chain:
```python
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_db.as_retriever()
)
answer = qa_chain.invoke({"query": "Explain cross validation"})
print(answer)
```